In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

# Path to the CSV file containing the articles
articles_df_deep_Cleaned = r'FULL PATH'

# Reading the CSV file into a DataFrame
df = pd.read_csv(articles_df_deep_Cleaned, encoding="utf-8")

# Converting 'Date Release' to datetime format
df['Date Release'] = pd.to_datetime(df['Date Release'], format='%Y-%m-%d')

# Removing duplicate articles based on 'ID'
df = df.drop_duplicates(subset=['ID'])

# Initializing a TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 1), min_df=0.0008, max_df=0.9)

# Fitting the vectorizer to the 'Paragraph Stripped' column of the DataFrame
vectorizer.fit(df['Paragraph Stripped'])

# Transforming the 'Paragraph Stripped' column
vectorizer_lookup = vectorizer.transform(df['Paragraph Stripped'])

# Parameters for the sigmoid functions
sigmmoid_paramters = [41.37306035617227, 0.15530631014140142, 40.529705247734285, 0.16170874090498477]

# Defining two sigmoid functions
def sigmoid_1(x):
    return 1 / (1 + np.exp(-sigmmoid_paramters[0] * (x - sigmmoid_paramters[1])))

def sigmoid_2(x):
    return 1 / (1 + np.exp(-sigmmoid_paramters[2] * (x - sigmmoid_paramters[3])))


In [22]:
def find_next_step(current_id_list, current_score, look_ahead):
    # Get the release date of the last article in the current chain
    date_for_article = df.loc[df['ID'] == current_id_list[-1], 'Date Release'].values[0]

    # Find the index of the last article in the DataFrame
    chosen_article_text = df[df["ID"] == current_id_list[-1]].index.values[0]
    # Retrieve its vector representation
    chosen_article = vectorizer_lookup[chosen_article_text]

    # Filter the DataFrame to include only articles released before the current article
    filtered_df = df[df['Date Release'] < date_for_article]      
    # If no previous articles are found, return None
    if len(filtered_df) == 0:
        return [None]

    # Retrieve vector representations for all filtered articles
    index_values = filtered_df.index.values
    vectorizer_lookup_rows = vectorizer_lookup[index_values]

    # Get the index and vector representation of the first article in the chain
    root_article_index = df[df["ID"] == current_id_list[0]].index.values[0]
    root_article = vectorizer_lookup[root_article_index]

    # Get IDs for potential next articles
    id_for_sample = filtered_df["ID"].values

    # Calculate cosine similarity with the first and last articles
    cos_result_start = cosine_similarity(root_article, vectorizer_lookup_rows)
    cos_result_in = cosine_similarity(chosen_article, vectorizer_lookup_rows) 

    # Evaluate and score potential next articles
    results = []
    for i, _ in enumerate(cos_result_in[0]):
        in_between_sim = cos_result_in[0][i]
        starting_sim = cos_result_start[0][i]
        # Compute new score using sigmoid functions and weights
        new_score = 0.8 * sigmoid_1(in_between_sim) + 0.2 * sigmoid_2(starting_sim) + current_score

        # Create a candidate chain by adding the new article
        candiate_id_list = current_id_list + [id_for_sample[i]]
        combined_list = [candiate_id_list, new_score]    
        results.append(combined_list)

    # Sort and limit the results if lookahead is enabled
    if look_ahead:
        results.sort(key=lambda x: x[1], reverse=True)
        results = results[:3]

    return results


def look_ahead(node, next_steps_fn, current_depth, max_total_depth, lookahead_depth):
    # If the maximum depth is reached or no lookahead is needed
    if current_depth >= max_total_depth or lookahead_depth == 0:
        return 0

    best_score = 0
    score = node[1]

    # Explore the next depth level
    next_depth = current_depth + 1
    for next_node in next_steps_fn(node[0], node[1], True):
        if next_node is None:
            continue

        # Calculate the future score recursively with reduced lookahead depth
        next_lookahead_depth = min(lookahead_depth - 1, max_total_depth - next_depth)
        future_score = look_ahead(next_node, next_steps_fn, next_depth, max_total_depth, next_lookahead_depth)
        best_score = max(best_score, score + future_score)

    return best_score


def beam_search_with_look_ahead(initial_state, branches, next_steps_fn, max_depth, lookahead_depth=3):
    # Initialize the beam with the initial state
    beam = [([initial_state], 0, 0)]

    # Iterate through each depth level up to the maximum depth
    for depth in range(max_depth):
        candidates = []
        for current_path, score, _ in beam:
            # Explore next steps for the current path
            for next_node in next_steps_fn(current_path, score, False):
                if next_node is None:
                    continue

                # Evaluate immediate and future potential scores
                immediate_score = next_node[1]
                current_path, score = next_node
                effective_lookahead_depth = min(lookahead_depth, max_depth - (depth + 1))
                future_potential = look_ahead(next_node, next_steps_fn, depth, max_depth, effective_lookahead_depth)
                total_score = future_potential

                # Add candidate paths to the list
                candidates.append((next_node[0], immediate_score, total_score))

        # Sort candidates by


In [26]:
# Inverting the DataFrame based on 'Date Release'
df_invert = df.sort_values(by=['Date Release'], ascending=False)

# Print the first 20 IDs for analysis
print(df_invert["ID"].values[:20])

# Applying the beam search with lookahead on the first 20 IDs
results_list = []
for id_value in df_invert["ID"].values[:20]:
    results = beam_search_with_look_ahead(id_value, 3, find_next_step, 4, vectorizer)
    print("done")
    results_list.append(results[0])

# Displaying results
for i in results_list:
    ids = i[0]
    score = i[1]
    print("----------------")
    print(f"ids: {ids}")
    print(f"score: {score}")
    # Retrieving headlines and dates for the identified article IDs
    results_headlines = df_invert[df_invert['ID'].isin(ids)]['Headline'].values
    results_dates = df_invert[df_invert['ID'].isin(ids)]['Date Release'].values
    # Displaying headlines and dates
    for headline, date in zip(results_headlines, results_dates):
        print("Headline:", headline)
        print("Date Released:", date)
        print("")
    print("----------------")

[1761278 1760892 1761047 1761007 1761008 1761015 1761016 1761027 1761035
 1761036 1761037 1761038 1761048 1761001 1761053 1761057 1761058 1761059
 1761060 1761061]
done
done
